In [335]:
from LINZ.CORS_Timeseries import TimeseriesList, CoordApiTimeseries
from LINZ.Geodetic import GDB
from LINZ.Geodetic.Ellipsoid import GRS80

import datetime
import pandas as pd
import csv
import numpy as np
import requests
import json
from astropy.time import Time
import re

***Set up input parameters for regression calculation***  
You need to add:  
  * ***start date***: variable=start,   
  * ***end date***: variable=end,  
  * ***solution type***: variable=solutiontype,   
  * ***online coordinate converter coordsys code***: variable=itrf_input_occ_code,  
  * ***a description for the Landonline adjustment***: variable=lol_adj_desc,  
  * ***new coordinate NZGD2000 order***: variable=nzgd2000_order,  
  * ***new coordinate NZVD2016 order***: variable=nzvd2016_order,  
Also:    
* You can specify a list of codes in ***option 1***, or get all cors in ***option 2.***

In addition to these inputs, there are ***four tests*** that compare the new coordinate with the current GDB coordinate, and whether that difference is more than a threshold.  The thresholds are set in those code blocks below.  The first test is called: "***Test 1 horz change - change bigger than the value specified***".    
The variable for the threshold for each test is called: ***specified_value_test#***, where # is the number of the test.

In [336]:
#All normal user inputs should be in this box
start=datetime.datetime(2025,3,20) #format = YYYY,M(M),D(D)
end=datetime.datetime(2025,3,24) #format = YYYY,M(M),D(D)

solutiontype = 'd20f_52_code_B'   #This is the solution used in the daily processing
itrf_input_occ_code = "LINZ:ITRF2020_XYZ"  #this is the code to tell the Online Coordinate Converter the input coordsys of the conversions

lol_adj_desc = "2025.02.01 Private CORS Update (DefMod v20180701 ITRF2020@2023-07-01)" # Make sure it has enough info so future users can find the misc job folder for this calculation
nzgd2000_order = "3" # The order for the NZGD2000 output coordinates
nzvd2016_order = "1V" # The older for the NZVD2016 output coordinates


In [337]:
#Eliot Sinclair
#No issues good to go
codes=['ESBR','ESGM','ESKV','ESNL','ESPM','ESQT','ESRN','ESTR']
#Inactive codes=['ESCM']

In [338]:
### Print out a list of important parameters based on the input.

crd_epoch = end + (start - end) / 2  #this is the epoch all the resulting coordinates will be in.  I like it to be 1 Jan, but you could pick any date.
print (f'crd_epoch: {crd_epoch} --- nominal coordinate epoch of the output coordinates\n')
#input_date =  datetime.datetime(2007, 4, 14, 11, 42, 50)

astropy_time_object = Time(crd_epoch,format='datetime')
crd_epoch_decimal_year = astropy_time_object.decimalyear
print (f'crd_epoch_decimal_year : {crd_epoch_decimal_year} --- as above but in decimal year format used in Online Coordinate Converter conversions\n')

ndays = (end-start).days
print (f'ndays: {ndays} --- the number of days from start date to end date\n')
print (f'ndays/2: {int(ndays/2)} --- every mark must have at least this many days of data\n') 
print (f'ndays/8:  {int(ndays/8)} --- every mark must have at least this many days of data before and after the crd_epoch\n')
#print("crd_epoch : " + crd_epoch.strftime("%d/%m/%Y %H:%M") + ' --- this is the nominal epoch of the calculated coordinates and the mid-date\n')

count = len(codes)
print (f'number of cors: {count} \n')
#for c in codes:
#    print (c, type(c))

crd_epoch: 2025-03-22 00:00:00 --- nominal coordinate epoch of the output coordinates

crd_epoch_decimal_year : 2025.2191780821918 --- as above but in decimal year format used in Online Coordinate Converter conversions

ndays: 4 --- the number of days from start date to end date

ndays/2: 2 --- every mark must have at least this many days of data

ndays/8:  0 --- every mark must have at least this many days of data before and after the crd_epoch

number of cors: 8 



In [339]:
### Set up dataframe to receive regression calculation error messages

df_regres_error = pd.DataFrame (codes, columns = ['siteid'])
df_regres_error['has_data'] = None
df_regres_error['calcd_crd'] = None
df_regres_error['comment'] = None
#df_regres_error

In [340]:
df_model_ITRF_crds = pd.DataFrame()

***Calculate coordinates***
1. creates a timeseries for each cors
2. removes outliers
3. finds marks with at least ndays/2 of data, and ndays/4 of data before and after crd_epoch
4. calculate regression
5. if marks don't have any coordinates or not enough coordinates or failed to calculate the regression: capture error message

In [341]:
for code in codes:
    #print (code)


    try:
        print (code, solutiontype)
        ts=CoordApiTimeseries(code,solutiontype = solutiontype,after=start,before=end) #this creates a timeseries object
    except:
        print ("timeseries object failed to create")
    try:
        #print (ts)
        dates,xyz=ts.withoutOutliers().getObs(enu=False)                                  #this removes outliers
        #print (xyz)

        # This gives every day from the start day a consecutive number starting from 1
        # It counts how many days of data there are and how many days before and after the mid-epoch
        num_day = (dates-start).astype('timedelta64[D]').astype(int) 
        num_day_size = num_day.size
        days_after_epoch = num_day[num_day > ndays/2].size
        days_before_epoch = num_day[num_day < ndays/2].size

        # if the number of days exceeds the arbitarily chosen values, then the regression coordinates are calculated
        if num_day_size > ndays/2 and days_after_epoch > ndays/8 and days_before_epoch > ndays/8:
            #print ('passes count')
            # Calculate regression and coordinates
            # 'dayno' is basically the date of each coordinate.  
            dayno = num_day.reshape((num_day.size,1)) #changed from 1d to 2d array where the second d is 1
            n = np.size(dayno)
            dayno_mean = dayno.mean()
            xyz_mean = xyz.mean(axis=0)  # This is an array of size 3.
            #print (xyz_mean)
            Sxy = np.sum(dayno*xyz, axis=0)- n*dayno_mean*xyz_mean
            Sxx = np.sum(dayno*dayno, axis=0)-n*dayno_mean*dayno_mean

            # b1 = slope
            b1 = Sxy/Sxx

            # b0 = y-intercept
            b0 = xyz_mean-b1*dayno_mean

            y_pred = b1 * (ndays/2) + b0
            new_row = pd.DataFrame([[code,*y_pred]],columns=['code','x','y','z'])# *y_pred each of the values in this iterable thing
            #print (new_row)
            df_model_ITRF_crds = pd.concat([df_model_ITRF_crds, new_row], ignore_index=True)
            df_regres_error.loc[df_regres_error.siteid == code, 'calcd_crd'] = 'y'
            df_regres_error.loc[df_regres_error.siteid == code, 'has_data'] = 'y'

        # if anything above fails it gets picked up in this 'else' or the 'except' below
        else:
            #print (f'not enough coordinates for station {code}')
            df_regres_error.loc[df_regres_error.siteid == code, 'calcd_crd'] = 'n'
            df_regres_error.loc[df_regres_error.siteid == code, 'has_data'] = 'y'
            df_regres_error.loc[df_regres_error.siteid == code, 'comment'] = f'not enough coordinates for station {code}'

    except Exception as ex:
        #print (f"couldn't get coords for station {code} {ex}")
        df_regres_error.loc[df_regres_error.siteid == code, 'has_data'] = 'n'
        df_regres_error.loc[df_regres_error.siteid == code, 'calcd_crd'] = 'n'
        df_regres_error.loc[df_regres_error.siteid == code, 'comment'] = f"couldn't get coords for station {code} {ex}"
        continue

ESBR d20f_52_code_B
ESGM d20f_52_code_B
ESKV d20f_52_code_B
ESNL d20f_52_code_B
ESPM d20f_52_code_B
ESQT d20f_52_code_B
ESRN d20f_52_code_B
ESTR d20f_52_code_B


In [342]:
##Send regression error information to csv

df_regres_error=df_regres_error.sort_values(['calcd_crd','has_data'])
df_regres_error.to_csv('regres_error_info.csv')
df_regres_error

,siteid,has_data,calcd_crd,comment
4,ESPM,n,n,couldn't get coords for station ESPM index 0 i...
0,ESBR,y,y,None
1,ESGM,y,y,None
2,ESKV,y,y,None
3,ESNL,y,y,None
5,ESQT,y,y,None
6,ESRN,y,y,None
7,ESTR,y,y,None


In [343]:
print (df_model_ITRF_crds)

   code             x              y             z
0  ESBR -4.593460e+06  588296.352600 -4.371063e+06
1  ESGM -4.656150e+06  721905.147400 -4.284436e+06
2  ESKV -4.628401e+06  582556.273867 -4.335490e+06
3  ESNL -4.763458e+06  566822.912367 -4.189381e+06
4  ESQT -4.430152e+06  880725.635133 -4.488615e+06
5  ESRN -4.584753e+06  613237.841067 -4.376791e+06
6  ESTR -4.592307e+06  595895.745800 -4.371256e+06


In [344]:
##Send global CORS ITRF calculated coordinates to csv

df_model_ITRF_crds.to_csv('global_cors_itrf_calc_coords.csv', index=False)

In [345]:
##Get a list of all marks we have ITRF2014 modelled coordinates for ready to query the GDB

mark_list = df_model_ITRF_crds['code'].tolist()
#print (mark_list)

***Get the GDB coordinates for all the marks in the mark_list and put in dataframe df_gdb_nzgd2000.***  
The mark must be a NEWZ mark, so overseas ones get culled at this stage, but it doesn't know if the overseas code is the same as an existing code on a different mark in NZ, but the coordinate difference should be massive so obvious.

In [346]:
columns=["code","gdb_long","gdb_lat","gdb_ellhgt"]
gdbdata=[]
for m in mark_list:
    #print (m)
    try:
        mark=GDB.get(m)
        c=mark.coordinate
        if mark.mark_data.country != 'NEWZ':
            continue
        gdbdata.append([m,c.longitude,c.latitude,c.height])
    except Exception as ex:
        #gdbdata.append([m,0.0,0.0,0.0,False])
        print (f"couldn't get coords for station {m} {ex}")
        continue
        
# df has the same rows as df, including isvalid column identifying which we are interested in
df_gdb_nzgd2000=pd.DataFrame(gdbdata,columns=columns)
df_gdb_nzgd2000

,code,gdb_long,gdb_lat,gdb_ellhgt
0,ESBR,172.701722,-43.538303,23.0634
1,ESGM,171.186842,-42.471965,28.2820
2,ESKV,172.826170,-43.095774,312.3520
3,ESNL,173.214054,-41.322194,30.5984
4,ESQT,168.756061,-45.012785,371.5528
5,ESRN,172.381580,-43.609129,62.4896
6,ESTR,172.606647,-43.540626,31.6526


In [347]:
occ_api = "https://www.geodesy.linz.govt.nz/api/conversions/v1/convert-to" 

input_crds = df_model_ITRF_crds[['x', 'y', 'z']].values.tolist() #This makes a list of x,y,z cooordinate lists for input so API is queried once.

crds_list=[]
print (crd_epoch_decimal_year, "is the epoch of the conversion")
coordlist = {
"crs": itrf_input_occ_code,
"coordinateOrder": ["geocentricX","geocentricY","geocentricZ"],
"coordinateEpoch": crd_epoch_decimal_year,
"coordinates": input_crds,
}
params = {"crs": "LINZ:NZGD2000" }

response = requests.post(occ_api, params=params, json=coordlist)
if response.status_code == 200:
    converted = response.json()
    crds_list = converted['coordinateList']['coordinates']

    #print(json.dumps(nzgdcoords, indent=2))
else:
    print(f"Error: {response.text}")

2025.2191780821918 is the epoch of the conversion


In [348]:
zipped_list = []
for code,crd in zip(df_model_ITRF_crds.code,crds_list): 
    if crd is not None:
        zipped_list.append([code,crd[0],crd[1],crd[2]])
        #print (idx,code,crd)
        
df_regres_converted_to_nzgd2000 = pd.DataFrame(zipped_list,columns=("code","mlat","mlong","mellhgt"))     

#with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
#    display(df_regres_converted_to_nzgd2000)
    
    
#df_regres_converted_to_nzgd2000  

In [349]:
df_combine = pd.merge(df_gdb_nzgd2000, df_regres_converted_to_nzgd2000, how='inner', left_on = 'code', right_on = 'code')
#df_combine

In [350]:
### Note: You can do these calcs more like (but thinking you want to see computation steps)
### Also this is quick enough there is no reason to use more approximate method below.
### Useful for understanding and how to quickly approximate, but not necessary in this case.

dedlon,dndlat=GRS80.metres_per_degree(df_combine.gdb_long,df_combine.gdb_lat)
df_combine['d_east']=(df_combine.mlong-df_combine.gdb_long)*dedlon
df_combine['d_north']=(df_combine.mlat-df_combine.gdb_lat)*dndlat
df_combine['d_horz']=np.sqrt(((df_combine.mlong-df_combine.gdb_long)*dedlon)**2 + ((df_combine.mlat-df_combine.gdb_lat)*dndlat)**2)
df_combine['d_ellhgt']=df_combine.mellhgt-df_combine.gdb_ellhgt
df_combine

,code,gdb_long,gdb_lat,gdb_ellhgt,mlat,mlong,mellhgt,d_east,d_north,d_horz,d_ellhgt
0,ESBR,172.701722,-43.538303,23.0634,-43.538302,172.701722,23.0490,-0.008392,0.002333,0.008711,-0.0144
1,ESGM,171.186842,-42.471965,28.2820,-42.471965,171.186842,28.3091,0.015867,-0.003814,0.016319,0.0271
2,ESKV,172.826170,-43.095774,312.3520,-43.095774,172.826170,312.3427,-0.006920,0.003333,0.007681,-0.0093
3,ESNL,173.214054,-41.322194,30.5984,-41.322194,173.214055,30.6201,0.038829,0.015771,0.041910,0.0217
4,ESQT,168.756061,-45.012785,371.5528,-45.012785,168.756061,371.5778,0.005912,-0.004556,0.007464,0.0250
5,ESRN,172.381580,-43.609129,62.4896,-43.609129,172.381580,62.5020,-0.007019,0.000679,0.007052,0.0124
6,ESTR,172.606647,-43.540626,31.6526,-43.540626,172.606647,31.6588,-0.003713,0.002241,0.004337,0.0062


In [351]:
specified_value_test1 = 0.025
df_combine[abs(df_combine.d_horz) > specified_value_test1]

,code,gdb_long,gdb_lat,gdb_ellhgt,mlat,mlong,mellhgt,d_east,d_north,d_horz,d_ellhgt
3,ESNL,173.214054,-41.322194,30.5984,-41.322194,173.214055,30.6201,0.038829,0.015771,0.04191,0.0217


In [352]:
specified_value_test2 = 0.025
df_combine[abs(df_combine.d_ellhgt) > specified_value_test2]

,code,gdb_long,gdb_lat,gdb_ellhgt,mlat,mlong,mellhgt,d_east,d_north,d_horz,d_ellhgt
1,ESGM,171.186842,-42.471965,28.2820,-42.471965,171.186842,28.3091,0.015867,-0.003814,0.016319,0.0271
4,ESQT,168.756061,-45.012785,371.5528,-45.012785,168.756061,371.5778,0.005912,-0.004556,0.007464,0.0250


In [353]:
specified_value_test3 = 0.05
df_combine[(abs(df_combine['d_ellhgt']) > specified_value_test3) & (abs(df_combine['d_horz']) > specified_value_test3)]

,code,gdb_long,gdb_lat,gdb_ellhgt,mlat,mlong,mellhgt,d_east,d_north,d_horz,d_ellhgt


In [354]:
specified_value_test4 = 0.05
df_combine[(abs(df_combine['d_ellhgt']) > specified_value_test4) | (abs(df_combine['d_horz']) > specified_value_test4)]

,code,gdb_long,gdb_lat,gdb_ellhgt,mlat,mlong,mellhgt,d_east,d_north,d_horz,d_ellhgt


In [355]:
df_combine.to_csv('df_combined_gdb_modelled.csv', index=False)
#df_model_ITRF_crds

In [356]:
df_model_ITRF_crds_list = df_model_ITRF_crds.values.tolist()
gdb_code_list = df_combine['code'].tolist()
#df_model_ITRF_crds_list

## ***Create crd file for fixed control for NGA***  
***Only create this coordinate file for the NGA for marks you want to use as fixed control***

In [358]:
#This section creates the header information for the SNAP crd file.
doy = '%Y:%j'
ymd = '%Y%m%d'

#print (type(crd_epoch))
print ("crd_epoch:",crd_epoch)

crd_epoch_ymd = datetime.datetime.strftime(crd_epoch, ymd)
print ("crd_epoch_ymd:",crd_epoch_ymd)

start_doy = datetime.datetime.strftime(start, doy)
end_doy = datetime.datetime.strftime(end, doy)
today = datetime.date.today()
print ("today:",today)

#create name for snap_crds file = '20230702_d20f_52_code_A_xyz.crd'
snap_crds = '{}_{}_xyz.crd'.format (crd_epoch_ymd, solutiontype)
print ("output snap crd file name: ", snap_crds)

itrf = re.split('_|:', itrf_input_occ_code)[1]
crd_heading = 'PositioNZ coordinates calculated on {} by averaging daily solutions from {} to {} @nominal epoch {}\n'.format (today, start_doy, end_doy, crd_epoch_ymd)
crd_system =  '{}_XYZ@{}\n'.format(itrf,crd_epoch_ymd)
crd_options = 'options no_geoid\n'
print (crd_heading)
print (crd_system)
print (crd_options)

#This section writes the SNAP crd file that contains the mean coordinates of the period queried.
with open(snap_crds, 'w')as out2:
    out2.write(crd_heading)
    out2.write(crd_system)
    out2.write(crd_options)
    out2.write("\n")
    
    for code_xyz in df_model_ITRF_crds_list:
        list_code = code_xyz[0]
        list_x = "{:.5f}".format(code_xyz[1])
        list_y = "{:.5f}".format(code_xyz[2])
        list_z = "{:.5f}".format(code_xyz[3])
        if list_code in gdb_code_list:
            out2.write('{}  {}  {}  {}\n'.format(list_code,list_x,list_y,list_z))

crd_epoch: 2025-03-22 00:00:00
crd_epoch_ymd: 20250322
today: 2025-05-09
output snap crd file name:  20250322_d20f_52_code_B_xyz.crd
PositioNZ coordinates calculated on 2025-05-09 by averaging daily solutions from 2025:079 to 2025:083 @nominal epoch 20250322

ITRF2020_XYZ@20250322

options no_geoid



## ***Convert ITRF XYZ coordinates to NZGD2000 lat/long and NZVD2016 hgt***  
***Then it outputs a dataframe, df_regres_converted_to_nzvd2016, that is in the same format as the Landonline load_coordinate_csv format.***

In [368]:
#df_regres_converted_to_nzgd2000

occ_api = "https://www.geodesy.linz.govt.nz/api/conversions/v1/convert-to" 

input_crds = df_model_ITRF_crds[['x', 'y', 'z']].values.tolist() #This makes a list of x,y,z cooordinate lists for input so API is queried once.

crds_list_v=[]

coordlist = {
"crs": itrf_input_occ_code,
"coordinateOrder": ["geocentricX","geocentricY","geocentricZ"],
"coordinateEpoch": crd_epoch_decimal_year,
"coordinates": input_crds,
}
params = {"crs": "LINZ:NZGD2000/NZVD2016"}

response = requests.post(occ_api, params=params, json=coordlist)
if response.status_code == 200:
    converted = response.json()
    crds_list_v = converted['coordinateList']['coordinates']

    #print(json.dumps(nzgdcoords, indent=2))
else:
    print(f"Error: {response.text}")
zipped_list_v = []
for code,crd in zip(df_model_ITRF_crds.code,crds_list_v): 
    #print (code,crd)
    if crd is not None:
        #zipped_list_v.append([code,crd[0],crd[1],crd[2]])
        zipped_list_v.append([code,crd[2]])
        #print (idx,code,crd)
        
df_regres_converted_to_nzvd2016 = pd.DataFrame(zipped_list_v,columns=("code","hgt"))     

#with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
#    display(df_regres_converted_to_nzgd2000)
    
#df_regres_converted_to_nzvd2016['markid']=""
df_regres_converted_to_nzvd2016.insert(1, "markid", np.nan)
df_regres_converted_to_nzvd2016.insert(2, "crdsys", "NZVD2016")
df_regres_converted_to_nzvd2016.insert(3, "lon", np.nan)
df_regres_converted_to_nzvd2016.insert(4, "lat", np.nan)
df_regres_converted_to_nzvd2016.insert(6, "order", nzvd2016_order)
df_regres_converted_to_nzvd2016.insert(7, "description", lol_adj_desc)

#df_regres_converted_to_nzvd2016     

In [369]:
df_horz_crds_lol=df_regres_converted_to_nzgd2000[['code','mlat','mlong','mellhgt']].copy()

df_horz_crds_lol.insert(1, "markid", np.nan)
df_horz_crds_lol.insert(2, "crdsys", "NZGD2000")
df_horz_crds_lol.insert(6, "order", nzgd2000_order)
df_horz_crds_lol.insert(7, "description", lol_adj_desc)

df_horz_crds_lol = df_horz_crds_lol.rename(columns={'mlat': 'lat', 'mlong': 'lon', 'mellhgt': 'hgt'})
df_horz_crds_lol = df_horz_crds_lol[['code','markid','crdsys','lon','lat','hgt','order','description']]

df_output_to_load_to_lol = pd.concat([df_horz_crds_lol,df_regres_converted_to_nzvd2016])  
df_output_to_load_to_lol.to_csv('df_output_to_load_to_lol.csv', index=False)
df_output_to_load_to_lol

,code,markid,crdsys,lon,lat,hgt,order,description
0,ESBR,NaN,NZGD2000,172.701722,-43.538302,23.0490,3,2025.02.01 Private CORS Update (DefMod v201807...
1,ESGM,NaN,NZGD2000,171.186842,-42.471965,28.3091,3,2025.02.01 Private CORS Update (DefMod v201807...
2,ESKV,NaN,NZGD2000,172.826170,-43.095774,312.3427,3,2025.02.01 Private CORS Update (DefMod v201807...
3,ESNL,NaN,NZGD2000,173.214055,-41.322194,30.6201,3,2025.02.01 Private CORS Update (DefMod v201807...
4,ESQT,NaN,NZGD2000,168.756061,-45.012785,371.5778,3,2025.02.01 Private CORS Update (DefMod v201807...
5,ESRN,NaN,NZGD2000,172.381580,-43.609129,62.5020,3,2025.02.01 Private CORS Update (DefMod v201807...
6,ESTR,NaN,NZGD2000,172.606647,-43.540626,31.6588,3,2025.02.01 Private CORS Update (DefMod v201807...
0,ESBR,NaN,NZVD2016,NaN,NaN,11.0628,1V,2025.02.01 Private CORS Update (DefMod v201807...
1,ESGM,NaN,NZVD2016,NaN,NaN,14.2840,1V,2025.02.01 Private CORS Update (DefMod v201807...
2,ESKV,NaN,NZVD2016,NaN,NaN,299.4398,1V,2025.02.01 Private CORS Update (DefMod v201807...


In [365]:
##Print marks in original codes list that didn't get coordinates in the output

code_list = df_output_to_load_to_lol['code'].tolist()
print (list(set(codes).difference(code_list)))

['ESPM']
